## 1. Importazioni / Imports

**In italiano:** Importiamo le librerie necessarie per il progetto. Usiamo `langchain_google_genai` per il modello linguistico, `DuckDuckGoSearchResults` per ottenere risultati di ricerca strutturati con link e snippet, `langgraph` per creare il nostro grafo di agenti e `transformers` con `torch` per l'inferenza del modello NLI.

**In English:** Import the necessary libraries for the project. We use `langchain_google_genai` for the language model, `DuckDuckGoSearchResults` to retrieve structured search results with links and snippets, `langgraph` to create our agent graph, and `transformers` with `torch` for NLI inference.


In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import DuckDuckGoSearchResults
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Any
from IPython.display import Image, display
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os


c:\Users\aless\Progetti Universitari\Sii\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Definizione degli Stati / State Definition

**In italiano:** Gli stati che verranno realizzati per il seguente progetto saranno 3:
* **InputState**: Uno stato di input, per permettere all'utente di inserire solo la query iniziale.
* **HiddenState**: Uno stato nascosto, che memorizza la query di ricerca raffinata, i risultati di ricerca come testo aggregato, i documenti recuperati con link, le fonti, il verdetto NLI, la confidence e la spiegazione generata.
* **OutputState**: Uno stato di output, che conterrà la risposta finale da mostrare all'utente insieme agli artefatti utili alla trasparenza.

**In English:** The states implemented for this project are 3:
* **InputState**: An input state to allow the user to provide only the initial query.
* **HiddenState**: A hidden state that stores the refined search query, the aggregated research text, the retrieved documents with links, the sources, the NLI verdict, the confidence, and the generated explanation.
* **OutputState**: An output state that holds the final response and the transparency artifacts returned to the user.


In [2]:
class InputState(TypedDict):
    query: str

class HiddenState(TypedDict):
    query: str
    search_query: str
    researches: str
    sources: list[str]
    retrieved_docs: list[str]
    nli_label: str
    confidence: float
    motivation: str
    response: str

class OutputState(TypedDict):
    researches: str
    sources: list[str]
    retrieved_docs: list[str]
    nli_label: str
    confidence: float
    response: str


## 3. Inizializzazione dei Modelli / Models Initialization

**In italiano:** Inizializziamo il modello LLM (`gemma-4-31b-it`) e il modello di Natural Language Inference (NLI) fine-tuned (DeBERTa). È importante ricordare che il dataset FEVER usa la codifica `SUPPORTS=0`, `NOT ENOUGH INFO=1`, `REFUTES=2`, mentre il modello base DeBERTa nasce tipicamente con l'ordine NLI standard `contradiction=0`, `entailment=1`, `neutral=2`. Per questo nel nodo NLI leggiamo la label reale dalla `config` del modello invece di usare un mapping hardcoded sugli indici.

**In English:** We initialize the LLM model (`gemma-4-31b-it`) and the fine-tuned Natural Language Inference (NLI) model (DeBERTa). It is important to remember that the FEVER dataset uses the encoding `SUPPORTS=0`, `NOT ENOUGH INFO=1`, `REFUTES=2`, while the base DeBERTa model typically follows the standard NLI order `contradiction=0`, `entailment=1`, `neutral=2`. For this reason, the NLI node reads the actual label from the model `config` instead of relying on a hardcoded index mapping.


In [ ]:
llm = ChatGoogleGenerativeAI(model='gemma-4-31b-it')

# Inizializza il modello NLI fine-tuned.
# Sostituire model_path con il percorso del proprio modello fine-tuned (es. '../fine-tuned_model').
model_path = "../fever-nli-deberta"
if not os.path.exists(model_path):
    print(f"Modello locale non trovato in {model_path}. Uso il modello base da HuggingFace come fallback.")
    model_path = "cross-encoder/nli-deberta-v3-large"

tokenizer = AutoTokenizer.from_pretrained(model_path)
nli_model = AutoModelForSequenceClassification.from_pretrained(model_path)
nli_model.eval()

# Imposta il device (GPU se disponibile, altrimenti CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nli_model.to(device)
print(f"Modello NLI caricato su: {device}")
print(f"id2label del modello: {getattr(nli_model.config, 'id2label', {})}")


Modello locale non trovato in ../fine-tuned_model. Uso il modello base da HuggingFace come fallback.


## 4. Definizione dei Nodi (Architettura RAG-NLI) / Nodes Definition

**In italiano:** Definiamo i nodi del grafo.
- `refine_query_node`: L'LLM estrae dalla notizia utente i concetti chiave per la ricerca sul web.
- `search_node`: Cerca le informazioni sul web e conserva anche snippet, link e documenti recuperati.
- `nli_classification_node`: Il modello NLI confronta l'evidenza trovata (Premessa) con la notizia dell'utente (Ipotesi) e converte la label reale del modello nel verdetto FEVER (`SUPPORTS`, `REFUTES`, `NOT ENOUGH INFO`).
- `generate_motivation_node`: L'LLM formula una risposta discorsiva per l'utente, basandosi sul risultato del modello NLI e allegando le fonti.

**In English:** Let's define the graph nodes.
- `refine_query_node`: The LLM extracts key concepts from the user's news for web search.
- `search_node`: Searches for information on the web and preserves snippets, links, and retrieved documents.
- `nli_classification_node`: The NLI model compares the found evidence (Premise) with the user's news (Hypothesis) and converts the model's real label into the FEVER verdict space (`SUPPORTS`, `REFUTES`, `NOT ENOUGH INFO`).
- `generate_motivation_node`: The LLM formulates a narrative response for the user, based on the NLI model's result and attaching sources.


In [ ]:
def _format_search_result(result: dict[str, str], index: int) -> str:
    title = result.get("title") or "Senza titolo"
    snippet = result.get("snippet") or "Snippet non disponibile."
    link = result.get("link") or "Link non disponibile."
    return f"[{index}] {title}\nSnippet: {snippet}\nLink: {link}"

def _normalize_search_results(results: Any) -> list[dict[str, str]]:
    if not isinstance(results, list):
        return []

    normalized_results: list[dict[str, str]] = []
    for item in results:
        if not isinstance(item, dict):
            continue

        normalized_results.append(
            {
                "title": str(item.get("title", "")).strip(),
                "snippet": str(item.get("snippet", "")).strip(),
                "link": str(item.get("link", "")).strip(),
            }
        )

    return normalized_results

def _extract_sources(results: list[dict[str, str]]) -> list[str]:
    seen: set[str] = set()
    sources: list[str] = []

    for result in results:
        link = result.get("link", "").strip()
        if not link or link in seen:
            continue
        seen.add(link)
        sources.append(link)

    return sources

def _extract_llm_text(response: Any) -> str:
    content = response.content
    if isinstance(content, list):
        content = "".join(
            part.get("text", "") if isinstance(part, dict) else str(part)
            for part in content
            if not isinstance(part, dict) or part.get("type") != "thinking"
        )
    return str(content).strip()

def _map_model_label_to_verdict(model_label: str) -> str:
    normalized_label = model_label.strip().lower().replace(" ", "_")
    verdict_map = {
        "entailment": "SUPPORTS",
        "supports": "SUPPORTS",
        "support": "SUPPORTS",
        "contradiction": "REFUTES",
        "refutes": "REFUTES",
        "refute": "REFUTES",
        "neutral": "NOT ENOUGH INFO",
        "not_enough_info": "NOT ENOUGH INFO",
        "nei": "NOT ENOUGH INFO",
    }

    if normalized_label not in verdict_map:
        raise ValueError(f"Etichetta NLI del modello non riconosciuta: {model_label}")

    return verdict_map[normalized_label]

def _resolve_model_label(predicted_class_id: int) -> str:
    id2label = getattr(nli_model.config, "id2label", {}) or {}
    model_label = id2label.get(predicted_class_id, id2label.get(str(predicted_class_id), ""))

    if model_label:
        return str(model_label)

    label2id = getattr(nli_model.config, "label2id", {}) or {}
    for label, class_id in label2id.items():
        if class_id == predicted_class_id:
            return str(label)

    raise ValueError(f"Impossibile risolvere la label NLI per class_id={predicted_class_id}")

def refine_query_node(state: HiddenState):
    prompt = f"""
    Devi fare una ricerca su Google/DuckDuckGo per verificare la seguente notizia: "{state['query']}"
    Estrai solo le parole chiave essenziali o formula una query di ricerca ottimale (max 5-6 parole).
    Rispondi unicamente con la stringa di ricerca, senza formattazione o introduzioni.
    """
    res = llm.invoke(prompt)
    search_query = _extract_llm_text(res)
    print(f"[Refine Node] Query di ricerca generata: {search_query}")
    return {"search_query": search_query}

def search_node(state: HiddenState):
    search = DuckDuckGoSearchResults(output_format="list", num_results=5)
    raw_results = search.invoke(state["search_query"])
    normalized_results = _normalize_search_results(raw_results)
    retrieved_docs = [
        _format_search_result(result, index)
        for index, result in enumerate(normalized_results, start=1)
    ]
    researches = "\n\n".join(retrieved_docs) if retrieved_docs else "Nessun risultato trovato."
    sources = _extract_sources(normalized_results)

    print(
        f"[Search Node] Risultati estratti: {len(normalized_results)} documenti, "
        f"{len(sources)} fonti."
    )
    return {
        "researches": researches,
        "retrieved_docs": retrieved_docs,
        "sources": sources,
    }

def nli_classification_node(state: HiddenState):
    premise = state["researches"]
    hypothesis = state["query"]

    inputs = tokenizer(
        premise,
        hypothesis,
        padding="max_length",
        truncation=True,
        max_length=256,
        return_tensors="pt",
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = nli_model(**inputs)

    logits = outputs.logits
    probs = torch.nn.functional.softmax(logits, dim=-1)
    predicted_class_id = logits.argmax().item()
    confidence = probs[0, predicted_class_id].item()
    model_label = _resolve_model_label(predicted_class_id)
    label = _map_model_label_to_verdict(model_label)
    print(
        f"[NLI Node] class_id={predicted_class_id}, model_label={model_label}, "
        f"verdict={label}, confidence={confidence}"
    )
    return {"nli_label": label, "confidence": confidence}

def generate_motivation_node(state: HiddenState):
    final_prompt = f"""
    Devi preparare il testo finale per un sistema di fact-checking.

    Claim dell'utente:
    "{state['query']}"

    Evidenze recuperate dal web:
    "{state['researches']}"

    Verdetto NLI:
    "{state['nli_label']}"

    Legenda del verdetto:
    - SUPPORTS = confermato dalle fonti
    - REFUTES = smentito dalle fonti
    - NOT ENOUGH INFO = dati insufficienti per una conclusione solida

    Scrivi in italiano una risposta professionale, chiara e logica.
    """
    res = llm.invoke(final_prompt)
    motivation_text = _extract_llm_text(res)
    final_response = (
        motivation_text
        + "\n\n---\n**Evidenza grezza recuperata dal web (DuckDuckGo):**\n"
        + state["researches"]
    )

    if state.get("sources"):
        final_response += "\n\n**Fonti:**\n" + "\n".join(
            f"- {source}" for source in state["sources"]
        )

    return {"motivation": motivation_text, "response": final_response}


## 5. Compilazione del Grafo / Graph Compilation

**In italiano:** Assembliamo il workflow lineare: estrazione query -> ricerca con fonti strutturate -> classificazione NLI -> generazione risposta.

**In English:** Assemble the linear workflow: query extraction -> structured web search -> NLI classification -> response generation.


In [ ]:
workflow = StateGraph(state_schema=HiddenState, input_schema=InputState, output_schema=OutputState)

workflow.add_node("refine_query", refine_query_node)
workflow.add_node("search_web", search_node)
workflow.add_node("nli_classification", nli_classification_node)
workflow.add_node("generate_motivation", generate_motivation_node)

workflow.add_edge(START, "refine_query")
workflow.add_edge("refine_query", "search_web")
workflow.add_edge("search_web", "nli_classification")
workflow.add_edge("nli_classification", "generate_motivation")
workflow.add_edge("generate_motivation", END)

app = workflow.compile()

In [ ]:
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print("Errore nella generazione dell'immagine. Verifica le dipendenze.")

## 6. Test e Invocazione / Test and Invocation

**In italiano:** Testiamo il nuovo grafo (RAG-NLI) passando una query.

**In English:** Let's test the new graph (RAG-NLI) by passing a query.

In [ ]:
input_data = {"query": "Addio Bastoni, netta posizione dell’Inter: annuncio di Romano"}
result = app.invoke(input_data)

print("\n========================== RISPOSTA FINALE ==========================\n")
print(result['response'])
print("\n============================== VERDETTO ==============================\n")
print(result['nli_label'], result['confidence'])
print("\n=============================== FONTI ===============================\n")
print(result['sources'])
